> 🔑 **SOLUTIONS NOTEBOOK**: instructor / self-check copy. Try the problems in `../notebooks/` first!

# 🔍 Lesson 08: Model Interpretability: Importance, Permutation & SHAP

**Time:** ~3 hours  **Prerequisites:** Lessons 02, 04, 06

We now know how to build an accurate ensemble and how to *prove* it is accurate. A stakeholder asks the next question:

> *"Fine. But **why** does it say this house is worth $920k?"*

Ensembles of hundreds of trees are **black boxes**: and "the model said so" is not an acceptable answer to a regulator, a customer, a doctor or your own debugging instincts.

**Why interpret models?**
* 🐛 **Debug:** catch leaks and silly shortcuts (*"it's predicting from the row ID?!"*).
* 🤝 **Trust & adoption:** people use models they understand.
* ⚖️ **Fairness & compliance:** show decisions don't rest on forbidden factors; explain individual decisions (e.g. credit).
* 🔬 **Discovery:** models often reveal structure in the data that surprises the experts.

### 🎯 Learning goals
1. Distinguish **global** (whole-model) from **local** (one-prediction) explanations.
2. Explain why **built-in impurity importance misleads**: and demonstrate it by planting a noise feature.
3. Implement and use **permutation importance**, and know its blind spot (correlated features).
4. Compute **Shapley values** by brute force on a small model, then use **TreeSHAP** on the real one.
5. Read **SHAP bar, beeswarm, dependence and waterfall plots**, and turn them into plain-English explanations.
6. State what interpretability methods **cannot** tell you (hint: causation).

| | **Global** | **Local** |
|---|---|---|
| question | *"Which features matter overall? How does the model respond to feature X?"* | *"Why this prediction for this house?"* |
| tools | importance, permutation importance, PDP, SHAP beeswarm/bar | SHAP waterfall / force, ICE, LIME |

In [1]:
import sys; sys.path.append("../utils")          # lets us import the course helpers
from course_utils import *                       # load_housing(), check(), set_style(), ...
import numpy as np, pandas as pd, matplotlib.pyplot as plt

set_style()
np.random.seed(SEED)
import shap, lightgbm as lgb
from itertools import combinations, permutations
from scipy.stats import spearmanr
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

## 0 · The model we will interrogate

A tuned-ish **LightGBM** on the housing data, using native categoricals and NaNs (Lesson 04). Everything below is evaluated on the **test set** so the explanations describe how the model behaves on unseen houses.

In [2]:
X_train, X_test, y_train, y_test = get_housing_split()

def to_cat(X, ref):
    X = X.copy()
    for c in HOUSING_CATEGORICAL:
        X[c] = pd.Categorical(X[c], categories=sorted(ref[c].dropna().unique()))
    return X
Xc_train, Xc_test = to_cat(X_train, X_train), to_cat(X_test, X_train)
def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))

model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15, subsample=0.8, subsample_freq=1,
                          colsample_bytree=0.8, random_state=SEED, verbose=-1).fit(Xc_train, y_train)
print(f"test RMSE = {rmse(y_test, model.predict(Xc_test)):.1f}   |   features: {list(Xc_train.columns)}")

test RMSE = 88.9   |   features: ['year_listed', 'neighborhood', 'property_type', 'size_sqm', 'bedrooms', 'bathrooms', 'age_years', 'renovated', 'floor', 'distance_to_center_km', 'transit_minutes', 'school_rating', 'security_rating', 'has_parking', 'has_garden', 'has_pool', 'energy_rating']


## 1 · Built-in feature importance: free, fast, and *dangerous*

Tree libraries report importance for free: **MDI** in random forests (mean decrease in impurity), **gain / split-count / cover** in XGBoost and LightGBM. It is computed from the *training* process: *how much did splits on feature $j$ reduce the loss, summed over all trees?*

### 🧪 Worked Example 8.1: plant a liar
Let's run an experiment. We take the housing features and **add two columns of pure garbage**:
* `random_noise`: a fresh standard-normal number per house;
* `random_code`: a random integer 0–999 (think "an ID-like column with lots of distinct values").

Neither has *anything* to do with price. A trustworthy importance measure must score them ≈ 0. We'll train a random forest and compare **MDI** with **permutation importance on the test set** (explained next; for now read it as "how much does test error get worse if I scramble this column?").

In [3]:
prep_num = make_preprocessor(X_train, onehot=False)                       # ordinal-coded categoricals -> plain numeric matrix
names = [n.split("__", 1)[1] for n in prep_num.fit(X_train).get_feature_names_out()]
rng = np.random.default_rng(0)
N_train = np.c_[prep_num.transform(X_train), rng.normal(size=len(X_train)), rng.integers(0, 1000, len(X_train))]
N_test  = np.c_[prep_num.transform(X_test),  rng.normal(size=len(X_test)),  rng.integers(0, 1000, len(X_test))]
names += ["random_noise", "random_code"]
N_train, N_test = pd.DataFrame(N_train, columns=names), pd.DataFrame(N_test, columns=names)

rf = RandomForestRegressor(n_estimators=300, max_features=0.4, random_state=SEED, n_jobs=-1).fit(N_train, y_train)
print(f"RF test RMSE: {rmse(y_test, rf.predict(N_test)):.1f}")

RF test RMSE: 129.0


In [4]:
from sklearn.inspection import permutation_importance
perm = permutation_importance(rf, N_test, y_test, n_repeats=5, random_state=0, scoring="neg_root_mean_squared_error", n_jobs=1)

cmp = pd.DataFrame({"MDI (built-in, train)": rf.feature_importances_, "permutation (test, RMSE increase)": perm.importances_mean}, index=names)
cmp["MDI rank"] = cmp["MDI (built-in, train)"].rank(ascending=False).astype(int)
cmp["perm. rank"] = cmp["permutation (test, RMSE increase)"].rank(ascending=False).astype(int)
cmp.sort_values("MDI rank").round(4).head(14)

,"MDI (built-in, train)","permutation (test, RMSE increase)",MDI rank,perm. rank
size_sqm,0.2965,151.1840,1,1
distance_to_center_km,0.1498,84.5084,2,2
bedrooms,0.1486,41.4981,3,3
school_rating,0.0746,37.3066,4,4
bathrooms,0.0689,8.8426,5,7
transit_minutes,0.0420,7.6849,6,8
has_pool,0.0322,19.2505,7,5
neighborhood,0.0273,10.3845,8,6
age_years,0.0256,4.6976,9,11
random_noise,0.0226,0.7177,10,15


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
order = cmp.sort_values("MDI rank").index[:12][::-1]
colors = [PALETTE["red"] if n.startswith("random") else PALETTE["blue"] for n in order]
axes[0].barh(order, cmp.loc[order, "MDI (built-in, train)"], color=colors); axes[0].set_title("Built-in (MDI) importance")
order2 = cmp.sort_values("perm. rank").index[:12][::-1]
colors2 = [PALETTE["red"] if n.startswith("random") else PALETTE["blue"] for n in order2]
axes[1].barh(order2, cmp.loc[order2, "permutation (test, RMSE increase)"], color=colors2); axes[1].set_title("Permutation importance (held-out)")
plt.suptitle("Red = columns of pure noise", y=1.0); plt.tight_layout(); plt.show()

### 🔎 The liar caught red-handed
Look at where the **red bars** (pure noise!) sit in the left chart: the built-in importance hands them **real, non-trivial scores**: in our run `random_noise` ranks **10th** and `random_code` **13th** out of 19 features, above genuine features such as `has_garden`, `has_parking` and `security_rating`. The permutation importance on held-out data drops them to **15th and 19th (the very bottom, ≈ 0)**, where they belong.

**Why does MDI lie?**
1. It's computed on **training data**: a forest happily makes splits on noise to memorise training rows, and each such split "reduces impurity" a little.
2. It's **biased toward continuous and high-cardinality features**, which offer many candidate thresholds: more chances to find a lucky split.
3. It tells you what the model *used*, not what *helped it generalise*.

> ⚠️ **Rule:** never present built-in importance as "what drives the outcome". Use it as a quick sketch; confirm with permutation importance or SHAP on **held-out** data.

## 2 · Permutation importance

**The idea (Breiman 2001; Fisher, Rudin & Dominici 2019):** *a feature is important if scrambling it hurts the model.*

1. Compute the model's score on a dataset (preferably held-out) → `baseline`.
2. For each feature $j$: **shuffle column $j$** (breaking its link to the target and to the other features), re-score.
3. Importance of $j$ = how much worse the score got (average over several shuffles).

It is **model-agnostic**, easy to explain, and measured on data the model hasn't memorised. Let's write it ourselves (it's 12 lines): we'll reuse it all lesson.

In [6]:
def perm_importance(model, X, y, groups=None, n_repeats=5, seed=0):
    """Increase in RMSE when a column (or a *group* of columns, shuffled together) is permuted."""
    rng = np.random.default_rng(seed)
    groups = groups or {c: [c] for c in X.columns}
    base = rmse(y, model.predict(X))
    out = {}
    for name, cols in groups.items():
        increases = []
        for _ in range(n_repeats):
            Xp = X.copy()
            order = rng.permutation(len(X))                      # one shuffled row order...
            for c in cols:
                Xp[c] = X[c].iloc[order].values                  # ...applied to every column of the group
            increases.append(rmse(y, model.predict(Xp)) - base)
        out[name] = float(np.mean(increases))
    return pd.Series(out).sort_values(ascending=False)

pi = perm_importance(model, Xc_test, y_test)
plt.figure(figsize=(8, 5)); pi.head(12)[::-1].plot.barh(color=PALETTE["blue"])
plt.xlabel("increase in test RMSE when the column is shuffled"); plt.title("Permutation importance; LightGBM on the test set"); plt.show()

Size, neighbourhood, distance and age dominate: in line with the recipe that generated the data. Features like `has_garden` barely register.

### The blind spot: correlated features
If two features carry the same information, scrambling **one** barely hurts: the model just reads the other. Each twin looks unimportant even though the *pair* is vital.

🔮 **Predict:** we'll add `size_sqft`: the same size in square feet plus a pinch of noise, and retrain. What happens to the importance of `size_sqm`?

In [7]:
rng = np.random.default_rng(1)
def add_twin(X): return X.assign(size_sqft=X["size_sqm"] * 10.764 + rng.normal(0, 2, len(X)))
Xtw_train, Xtw_test = add_twin(Xc_train), add_twin(Xc_test)
twin_model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, random_state=SEED, verbose=-1).fit(Xtw_train, y_train)

before = perm_importance(model, Xc_test, y_test)["size_sqm"]
after = perm_importance(twin_model, Xtw_test, y_test)
grouped = perm_importance(twin_model, Xtw_test, y_test, groups={"size (both columns together)": ["size_sqm", "size_sqft"]})

print(f"size_sqm importance, original model        : {before:6.1f}")
print(f"size_sqm importance, with a twin column    : {after['size_sqm']:6.1f}     size_sqft: {after['size_sqft']:6.1f}   <- split / masked")
print(f"both columns permuted together (grouped)   : {grouped['size (both columns together)']:6.1f}   <- true importance of 'size'")

size_sqm importance, original model        :  232.2
size_sqm importance, with a twin column    :   92.9     size_sqft:  115.6   <- split / masked
both columns permuted together (grouped)   :  275.2   <- true importance of 'size'


Adding the twin **splits the credit** (and masks it): each copy shows a fraction of the true importance. The cure is **grouped permutation**: shuffle correlated columns *together*: or drop/merge redundant features. Remember this whenever your importance chart "undervalues" something you know matters.

## 3 · Partial dependence & ICE: how does the prediction *respond*?

Importance says *how much* a feature matters, not *how*. **Partial dependence (PDP)** answers: "if I set feature $x_j$ to value $v$ for **every** house and average the predictions, what happens as $v$ changes?"
**ICE** curves draw the same thing for *individual* houses (one line each): if they all move together there are no interactions; if they fan out, there are.

In [8]:
def ice(model, X, feature, grid, n_lines=40, seed=0):
    rows = X.sample(n_lines, random_state=seed)
    curves = []
    for v in grid:
        Xv = rows.copy(); Xv[feature] = v
        curves.append(model.predict(Xv))
    return np.array(curves).T                                  # shape (n_lines, len(grid))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feature in zip(axes, ["size_sqm", "distance_to_center_km", "transit_minutes"]):
    grid = np.linspace(Xc_test[feature].quantile(0.02), Xc_test[feature].quantile(0.98), 25)
    curves = ice(model, Xc_test, feature, grid)
    ax.plot(grid, curves.T, color=PALETTE["gray"], alpha=.25, lw=1)
    ax.plot(grid, curves.mean(axis=0), color=PALETTE["red"], lw=3, label="partial dependence (average)")
    ax.set_xlabel(feature); ax.set_ylabel("predicted price"); ax.legend()
plt.suptitle("ICE lines (grey) and partial dependence (red)"); plt.tight_layout(); plt.show()

The `size_sqm` lines **fan out**: the effect of an extra square metre depends on the house (neighbourhood premium × size interaction). The `transit_minutes` curve has a visible **drop near 8 minutes**: remember that?
We'll come back to it with SHAP. PDPs are a great tool, but averaging over *unrealistic* combinations (e.g. a 600 m² apartment) and ignoring correlations are their weaknesses: one motivation for SHAP.

## 4 · Shapley values: fair credit, from game theory

Imagine three coworkers produce a bonus together, and you must split it fairly. Lloyd **Shapley** (Nobel 1953/2012) showed the *only* split satisfying four sensible fairness axioms is:

> Give each player their **average marginal contribution**, over **all possible orders** in which the players could have joined the team.

Translate to prediction: **players = features**, **the payout = the model's prediction** for one house minus the average prediction. A feature's Shapley value $\phi_j$ is how much it adds, on average, across all orders in which we "reveal" features one at a time.

For this we need a **value function** $v(S)$: *the model's expected prediction when only the features in set $S$ are known (the rest are averaged out over background data)*.

$$\phi_j = \frac{1}{M!}\sum_{\text{orderings}} \Big[ v(\text{features before } j \cup \{j\}) - v(\text{features before } j) \Big]\qquad\text{(equivalently, a weighted sum over subsets)}$$

### Properties (why people love SHAP)
* **Efficiency / additivity:** $\;\text{base value} + \sum_j \phi_j = \text{the model's prediction}$: the explanation *adds up exactly*.
* **Symmetry:** features that contribute equally get equal credit. **Dummy:** a feature that never changes the output gets 0.
* **Linearity** across models (so it works for ensembles by summing trees).

### 🧪 Worked Example 8.2: brute-force Shapley values for one house
To enumerate every ordering we must keep the feature count tiny, so we fit a small model on **three** features: `size_sqm`, `distance_to_center_km`, `school_rating`, and explain one house.

In [9]:
three = ["size_sqm", "distance_to_center_km", "school_rating"]
med = X_train[three].median()
S_train, S_test = X_train[three].fillna(med), X_test[three].fillna(med)
small = GradientBoostingRegressor(n_estimators=100, max_depth=3, random_state=SEED).fit(S_train, y_train)

background = S_train.sample(100, random_state=0).values      # the "average house" population
house = S_test.iloc[0]; x = house.values
print(house.to_dict(), "\n-> model prediction:", round(small.predict(x[None])[0], 1))

def coalition_value(model, x, background, S):
    """v(S): average prediction when features in S are fixed to this house's values and the others are drawn from the background."""
    Z = background.copy()
    for j in S:
        Z[:, j] = x[j]
    return model.predict(Z).mean()

print("\nv(empty set)  =", round(coalition_value(small, x, background, set()), 1), "  <- the base value: the average prediction")
print("v(all 3)      =", round(coalition_value(small, x, background, {0, 1, 2}), 1), "  <- this house's prediction")

{'size_sqm': 166.0, 'distance_to_center_km': 5.11, 'school_rating': 8.9} 
-> model prediction: 920.7

v(empty set)  = 486.4   <- the base value: the average prediction
v(all 3)      = 920.7   <- this house's prediction


In [10]:
rows = []
for order in permutations(range(3)):                           # all 3! = 6 orders in which features could be "revealed"
    known, contrib = set(), {}
    for j in order:
        before = coalition_value(small, x, background, known)
        known = known | {j}
        contrib[three[j]] = coalition_value(small, x, background, known) - before
    rows.append({"order": " → ".join(three[j].split("_")[0] for j in order), **contrib})
table = pd.DataFrame(rows)
print(table.round(1).to_string(index=False))
phi_brute = table[three].mean()
print("\nShapley value = average over the 6 orders:")
print(phi_brute.round(2).to_string())
print(f"\nbase {coalition_value(small, x, background, set()):.1f} + sum of Shapley values {phi_brute.sum():.1f} = {coalition_value(small, x, background, set()) + phi_brute.sum():.1f}   vs prediction {small.predict(x[None])[0]:.1f}")

                   order  size_sqm  distance_to_center_km  school_rating
size → distance → school     180.9                  118.3          135.2
size → school → distance     180.9                   58.3          195.1
distance → size → school     203.5                   95.7          135.2
distance → school → size     238.6                   95.7          100.0
school → size → distance     234.1                   58.3          141.9
school → distance → size     238.6                   53.8          141.9

Shapley value = average over the 6 orders:
size_sqm                 212.78
distance_to_center_km     80.03
school_rating            141.53

base 486.4 + sum of Shapley values 434.3 = 920.7   vs prediction 920.7


Each row shows a different "story" of how the features arrive, and the credit each gets **depends on the order** (size matters differently if you already know the neighbourhood-like `distance`). Averaging over all orders produces the unique fair split: and **the pieces add up exactly** to the prediction minus the base value.

Now the punchline: the `shap` library computes the same numbers (with its *interventional* mode, using the identical background data): in microseconds for tree models.

In [11]:
explainer_small = shap.TreeExplainer(small, data=background, feature_perturbation="interventional")
phi_shap = explainer_small.shap_values(x[None])[0]
print(pd.DataFrame({"brute force": phi_brute.values, "shap.TreeExplainer": phi_shap}, index=three).round(4))
print("max difference:", np.abs(phi_brute.values - phi_shap).max())

                       brute force  shap.TreeExplainer
size_sqm                  212.7837            212.7837
distance_to_center_km      80.0262             80.0262
school_rating             141.5334            141.5334
max difference: 2.3828186215268943e-06


**Identical.** There's no magic in SHAP: just Shapley values, computed efficiently.
Brute force needs $2^M$ coalitions (impossible beyond ~20 features). **TreeSHAP** (Lundberg et al., 2018) exploits the tree structure to compute *exact* Shapley values in polynomial time. That is what makes SHAP practical for XGBoost / LightGBM / CatBoost / random forests.

## 5 · SHAP on the real model

Let's explain all 600 test houses with the 17-feature LightGBM. (`TreeExplainer` with no background data uses the trees' own *path-dependent* approximation; slightly different from the interventional version, but fast and standard.)

In [12]:
explainer = shap.TreeExplainer(model)
sv = explainer(Xc_test)                                   # an Explanation object: .values (600 x 17), .base_values, .data
feature_names = list(Xc_test.columns)

pred = model.predict(Xc_test)
err = np.abs(sv.base_values + sv.values.sum(axis=1) - pred).max()
print(f"base value (average prediction): {sv.base_values[0]:.1f}")
print(f"largest additivity error over 600 houses: {err:.2e}   <- base + sum(SHAP) reproduces every prediction")

base value (average prediction): 451.7
largest additivity error over 600 houses: 5.68e-12   <- base + sum(SHAP) reproduces every prediction


### Global view 1: the bar plot (mean |SHAP|)
Average magnitude of each feature's contribution: a *global importance* that, unlike MDI, reflects **actual effect sizes in the model's output units ($k)**.

In [13]:
shap.plots.bar(sv, max_display=12, show=False)
plt.title("Mean |SHAP value|: average impact on the predicted price ($k)"); plt.tight_layout(); plt.show()

### Global view 2: the beeswarm: importance **and direction**
Each dot is one house. **x-position** = that feature's SHAP value (push up/down on the price); **colour** = the feature's own value (red high, blue low; grey = categorical).
Read `size_sqm`: red dots (big houses) sit far right (↑ price), blue dots (small) left (↓). `age_years`: red (old) is left: older houses are cheaper.

In [14]:
shap.plots.beeswarm(sv, max_display=12, show=False)
plt.tight_layout(); plt.show()

### Did the model discover the recipe?

Here synthetic data pays off: *we know the truth.* The data generator contains: a **step** in price at ≤ 8 minutes from a station; a **hinge**: school quality only adds value above 6; an **age** penalty that is **cancelled by renovation**; and a log-ish size effect. Plot each feature's SHAP value against its value (a **dependence plot**):

In [15]:
idx = {n: i for i, n in enumerate(feature_names)}
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

ax = axes[0, 0]; ax.scatter(Xc_test["size_sqm"], sv.values[:, idx["size_sqm"]], s=8, alpha=.5)
ax.set_xlabel("size_sqm"); ax.set_ylabel("SHAP value ($k)"); ax.set_title("Size: rising, saturating-ish")

ax = axes[0, 1]; ax.scatter(Xc_test["transit_minutes"], sv.values[:, idx["transit_minutes"]], s=8, alpha=.5, color=PALETTE["green"])
ax.axvline(8, color="k", ls=":"); ax.set_xlim(0, 40); ax.set_xlabel("transit_minutes"); ax.set_title("Transit: a STEP near 8 minutes (true: ≤ 8)")

ax = axes[1, 0]; ax.scatter(Xc_test["school_rating"], sv.values[:, idx["school_rating"]], s=8, alpha=.5, color=PALETTE["orange"])
ax.axvline(6, color="k", ls=":"); ax.set_xlabel("school_rating"); ax.set_title("Schools: flat below 6, rising above (true hinge at 6)")

ax = axes[1, 1]
reno = Xc_test["renovated"].values == 1
ax.scatter(Xc_test["age_years"][~reno], sv.values[~reno, idx["age_years"]], s=8, alpha=.5, label="not renovated", color=PALETTE["red"])
ax.scatter(Xc_test["age_years"][reno], sv.values[reno, idx["age_years"]], s=8, alpha=.5, label="renovated", color=PALETTE["teal"])
ax.set_xlabel("age_years"); ax.set_title("Age penalty is much weaker for renovated houses"); ax.legend()
for a in axes.ravel(): a.axhline(0, color="gray", lw=.6)
plt.tight_layout(); plt.show()

**The model recovered every planted structure** from data alone: the step at 8 minutes, the school-rating hinge at 6, the renovation-moderated age decay. That is the discovery use-case of interpretability.

And the categorical `neighborhood`? Its mean SHAP value per neighbourhood should line up with the *true* log-price premiums baked into the generator:

In [16]:
true_premium = {"Riverside": .35, "Hilltop": .55, "Old Town": .20, "Greenfield": .10, "Lakeview": .45, "Market District": .15,
                "Station Heights": .25, "Parkside": .30, "Harbor Point": .40, "Outer Meadows": -.15, "Industrial East": -.25, "Sunnydale": 0.0}
nb = pd.DataFrame({"neighborhood": Xc_test["neighborhood"].astype(str).values, "shap": sv.values[:, idx["neighborhood"]]})
mean_shap = nb.groupby("neighborhood")["shap"].mean()
rho = spearmanr([true_premium[n] for n in mean_shap.index], mean_shap.values).statistic

plt.figure(figsize=(7, 4.5))
plt.scatter([true_premium[n] for n in mean_shap.index], mean_shap.values, s=60, color=PALETTE["blue"])
for n, v in mean_shap.items(): plt.annotate(n, (true_premium[n], v), textcoords="offset points", xytext=(5, 4), fontsize=8)
plt.xlabel("TRUE log-price premium (hidden recipe)"); plt.ylabel("mean SHAP of `neighborhood` ($k)")
plt.title(f"The model re-learned the neighbourhood premiums  (Spearman ρ = {rho:.2f})"); plt.show()

### Local view: the waterfall: explaining *one* house
A waterfall starts at the **base value** (average prediction) and adds each feature's push up (red) or down (blue) until it reaches **this house's prediction** $f(x)$.

In [17]:
i = 0
shap.plots.waterfall(sv[i], max_display=10, show=False)
plt.tight_layout(); plt.show()
print(f"house #{i}: predicted ${pred[i]:,.0f}k   |   actual ${y_test.values[i]:,.0f}k")

house #0: predicted $647k   |   actual $510k


### From plot to words (the skill that gets you promoted)
Never hand a stakeholder a waterfall without a sentence. Template:

> *"The model prices this house at **$X k**, against an average of **$B k** in the dataset. The biggest **upward** drivers are **[feature = value] (+$a k)** and **[…]**. The biggest **downward** drivers are **[…] (−$d k)**. Taken together, the remaining features net out to **±$r k**."*

We can automate it:

In [18]:
def explain_in_words(i, k=3):
    contrib = pd.Series(sv.values[i], index=feature_names)
    vals = Xc_test.iloc[i]
    up, down = contrib.nlargest(k), contrib.nsmallest(k)
    shown = list(up.index) + list(down.index)
    fmt = lambda f: f"{f} = {vals[f]} ({contrib[f]:+.0f}k)"
    rest = contrib.drop(shown).sum()
    return (f"The model prices house #{i} at ${pred[i]:,.0f}k versus an average of ${sv.base_values[i]:,.0f}k.\n"
            f"  Biggest upward drivers  : " + "; ".join(fmt(f) for f in up.index if contrib[f] > 0) + "\n"
            f"  Biggest downward drivers: " + "; ".join(fmt(f) for f in down.index if contrib[f] < 0) + "\n"
            f"  All other features together: {rest:+.0f}k")
print(explain_in_words(0)); print(); print(explain_in_words(7))

The model prices house #0 at $647k versus an average of $452k.
  Biggest upward drivers  : size_sqm = 166.0 (+131k); school_rating = 8.9 (+80k); neighborhood = Parkside (+63k)
  Biggest downward drivers: year_listed = 2020 (-61k); age_years = 28.0 (-58k); bedrooms = 3 (-28k)
  All other features together: +69k

The model prices house #7 at $695k versus an average of $452k.
  Biggest upward drivers  : size_sqm = 192.0 (+204k); age_years = 1.0 (+69k); property_type = detached (+48k)
  Biggest downward drivers: transit_minutes = 11.0 (-48k); year_listed = 2021 (-26k); has_pool = 0 (-13k)
  All other features together: +9k


## 6 · Comparing the three global importance rankings

Do permutation importance and SHAP agree? Usually broadly, but each answers a slightly different question:

* **Permutation importance:** *"how much does **test error** grow without this feature's information?"* (depends on the metric; sensitive to correlations)
* **Mean |SHAP|:** *"how much does this feature **move the predictions**, on average?"* (a statement about the model's output, regardless of accuracy)

In [19]:
mean_abs_shap = pd.Series(np.abs(sv.values).mean(axis=0), index=feature_names)
imp_gain = pd.Series(model.booster_.feature_importance(importance_type="gain"), index=feature_names)
rank_table = pd.DataFrame({"LightGBM gain": imp_gain.rank(ascending=False), "permutation": pi.rank(ascending=False),
                           "mean |SHAP|": mean_abs_shap.rank(ascending=False)}).astype(int).sort_values("mean |SHAP|")
print("rank of each feature (1 = most important):")
rank_table.head(10)

rank of each feature (1 = most important):


,LightGBM gain,permutation,mean |SHAP|
size_sqm,1,1,1
neighborhood,2,2,2
distance_to_center_km,4,3,3
bedrooms,3,4,4
age_years,6,6,5
year_listed,8,5,6
school_rating,5,7,7
property_type,7,10,8
transit_minutes,10,11,9
renovated,12,8,10


The top of the three lists is the same set of features; the middle ranks reshuffle. **Disagreement in the middle is normal and informative**: when rankings disagree strongly, suspect correlated features, interactions, or a very flat importance profile.

## 7 · Interpretability as a debugging tool: catching a leak

*Setting:* a colleague proudly reports R² = 0.99. You ask for the importance chart. We simulate their model by adding a column `agent_valuation`: an estate agent's **valuation of the very house we are predicting**, close to the sale price (a classic leak: the feature is derived from the target or only exists after the outcome).

In [20]:
rng = np.random.default_rng(5)
leak = lambda yy: yy.values * np.exp(rng.normal(0, 0.04, len(yy)))
Xl_train, Xl_test = Xc_train.assign(agent_valuation=leak(y_train)), Xc_test.assign(agent_valuation=leak(y_test))
leaky_model = lgb.LGBMRegressor(n_estimators=400, learning_rate=0.05, num_leaves=15, random_state=SEED, verbose=-1).fit(Xl_train, y_train)

from sklearn.metrics import r2_score
print(f"test R² = {r2_score(y_test, leaky_model.predict(Xl_test)):.3f}   (suspiciously good!)")
leak_imp = perm_importance(leaky_model, Xl_test, y_test)
leak_imp.head(6).round(1)

test R² = 0.989   (suspiciously good!)


agent_valuation    453.6
neighborhood         2.5
size_sqm             1.4
year_listed          0.8
age_years            0.6
renovated            0.2
dtype: float64

One feature **towers** over all others and the model is "too good to be true". That's the signature of leakage: ask *"would this feature exist at prediction time? Is it derived from the target?"*: if not, remove it. **A suspiciously dominant feature is a bug until proven otherwise.**

## 8 · What interpretability can and cannot tell you

| ✅ SHAP / importance **can** say | ❌ …but they **cannot** say |
|---|---|
| how the **model** behaves and what drives *its* predictions | what **causes** the outcome in the real world |
| which features the model relies on, and in what direction | that changing a feature will change the outcome (correlation ≠ causation) |
| debug leaks, discover structure, communicate individual decisions | that the model is *right* or *fair*: a biased model gets a faithful, biased explanation |

**Further caveats**
* **Correlated features share credit** (SHAP splits it between them; permutation importance masks it): don't read a single feature's value in isolation.
* Explanations are **model-dependent**: retrain with another seed or algorithm and the story can shift. Check stability.
* **Interventional vs path-dependent** SHAP can differ with correlated features; know which your library uses.
* Explanations can be **gamed or misread**; for high-stakes decisions pair them with domain review and, where possible, inherently interpretable models.

### ✋ Check your understanding

Try to answer *out loud* before you open the answer. If you can't say it simply, you don't own it yet.

**Q1. Why can impurity-based (MDI) importance give a pure-noise column a high score?**

<details><summary>Show answer</summary>

It is computed on the **training** data, where a flexible model can find chance splits on noise that reduce training impurity, and continuous/high-cardinality columns offer many candidate thresholds to exploit. It measures use during training, not usefulness on new data.

</details>

**Q2. You duplicate a crucial feature and its permutation importance drops by half for each copy. Did the information become less useful?**

<details><summary>Show answer</summary>

No. Shuffling one copy leaves the other intact, so the model barely suffers: the *credit is shared/masked*. Permute the copies together (grouped permutation) or remove redundancy to see the real importance.

</details>

**Q3. State the efficiency property of Shapley values and why it matters for explanations.**

<details><summary>Show answer</summary>

base value + Σ φ_j = the model's prediction. It guarantees an explanation that *accounts for the entire difference* between this prediction and the average one, so nothing is left unexplained.

</details>

**Q4. A SHAP beeswarm shows `age_years` with red (old) dots on the left. What does that mean? Does it prove old age *causes* lower prices?**

<details><summary>Show answer</summary>

Old houses (red) have **negative** SHAP values: the model lowers its price prediction for them. It describes the *model's* behaviour. Causation isn't established: age may be correlated with other unobserved factors (e.g. location, condition), and the model only learned associations.

</details>

**Q5. Why is brute-force Shapley computation infeasible for 100 features, and how does TreeSHAP get around it?**

<details><summary>Show answer</summary>

There are 2^100 feature coalitions. TreeSHAP exploits the tree structure (the expected prediction for any coalition can be tracked along the tree paths) to compute exact Shapley values in polynomial time.

</details>

## 9 · Problems

Variables available: `model`, `Xc_test`, `y_test`, `sv` (SHAP Explanation), `feature_names`, `pred`, `perm_importance`, `rmse`, `small`, `background`, `x`, `three`, `coalition_value`, `leaky_model`, `Xl_test`.

### 🧩 Problem 08.1 ⭐: Rank by mean |SHAP|

From the Explanation `sv`:

* `top3`: a list of the **three feature names** with the largest mean absolute SHAP value (descending),
* `add_err`: the **largest absolute difference** over all test houses between `base value + sum of SHAP values` and the model's prediction `pred`.

(The additivity error should be tiny; a good sanity check for any explainer.)

<details><summary>💡 Hint 1</summary>

`np.abs(sv.values).mean(axis=0)` gives one number per feature (column order = `feature_names`).

</details>

<details><summary>💡 Hint 2</summary>

`pd.Series(that, index=feature_names).nlargest(3).index.tolist()`.

</details>

<details><summary>💡 Hint 3</summary>

`np.abs(sv.base_values + sv.values.sum(axis=1) - pred).max()`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [21]:
mean_abs = pd.Series(np.abs(sv.values).mean(axis=0), index=feature_names)
top3 = mean_abs.nlargest(3).index.tolist()
add_err = float(np.abs(sv.base_values + sv.values.sum(axis=1) - pred).max())

# ---- self-check ----
check("08.1", lambda: top3 == pd.Series(np.abs(sv.values).mean(axis=0), index=feature_names).nlargest(3).index.tolist()
      and add_err < 1e-6 and "size_sqm" in top3, "Use mean(|SHAP|) over houses (axis=0). size_sqm must be among the top three; additivity error < 1e-6.",
      needs=[top3, add_err])

✅ 08.1: correct -- nice work!


**✅ Solution walk-through (Problem 08.1)**

In our run the top three are `size_sqm` (mean |SHAP| ≈ $153k), `neighborhood` (≈ $80k) and `distance_to_center_km` (≈ $62k). The additivity error is ~1e-12 (floating-point noise) because TreeSHAP values are *exact* for the model.

### 🧩 Problem 08.2 ⭐⭐: Permutation importance for one feature

Without using `perm_importance`, compute the permutation importance of two features on the **test set** for `model`:

* `imp_size`: for `size_sqm`
* `imp_pool`: for `has_pool`

For each: copy `Xc_test`, shuffle that column with a fixed `np.random.default_rng(0)`, record the **increase in RMSE** relative to the unshuffled baseline, and average over 5 repeats.

<details><summary>💡 Hint 1</summary>

`base = rmse(y_test, model.predict(Xc_test))`.

</details>

<details><summary>💡 Hint 2</summary>

In each repeat: `Xp = Xc_test.copy(); Xp[col] = Xp[col].sample(frac=1, random_state=int(rng.integers(1_000_000))).values`; then `rmse(y_test, model.predict(Xp)) - base`.

</details>

<details><summary>💡 Hint 3</summary>

Average the 5 increases with `np.mean`. Expect `imp_size` in the tens-to-hundreds and `imp_pool` near 0–5.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [22]:
rng = np.random.default_rng(0)
base = rmse(y_test, model.predict(Xc_test))

def shuffled_increase(col, n_repeats=5):
    incs = []
    for _ in range(n_repeats):
        Xp = Xc_test.copy()
        Xp[col] = Xp[col].sample(frac=1, random_state=int(rng.integers(1_000_000))).values
        incs.append(rmse(y_test, model.predict(Xp)) - base)
    return float(np.mean(incs))

imp_size = shuffled_increase("size_sqm")
imp_pool = shuffled_increase("has_pool")

# ---- self-check ----
check("08.2", lambda: imp_size > 40 and imp_pool < 0.2 * imp_size and imp_pool > -5,
      "Shuffling size_sqm should hurt a lot (≥ 40 RMSE), shuffling has_pool very little.", needs=[imp_size, imp_pool])

✅ 08.2: correct -- nice work!


**✅ Solution walk-through (Problem 08.2)**

Shuffling `size_sqm` more than triples the error (≈ +230 RMSE on top of a baseline of ≈ 89), while `has_pool` (present in few houses, small effect) barely registers. This is exactly what `perm_importance` does for every column.

### 🧩 Problem 08.3 ⭐⭐⭐: Shapley values the *other* way: subset weights

In Worked Example 8.2 we averaged over **orderings**. The same Shapley value can be written as a weighted sum over **subsets** $S$ of the *other* features:

$$\phi_j=\sum_{S\subseteq N\setminus\{j\}}\frac{|S|!\,(M-|S|-1)!}{M!}\,\big[v(S\cup\{j\})-v(S)\big].$$

Implement `shapley_subsets(model, x, background, j)` using `itertools.combinations` and `coalition_value`, then compute `phi_dist`: the Shapley value of **`distance_to_center_km`** (feature index 1) for the *same* house `x`.
It must match the table from the lesson (`phi_brute`).

<details><summary>💡 Hint 1</summary>

`from math import factorial`; `M = len(x)`; others = `[i for i in range(M) if i != j]`.

</details>

<details><summary>💡 Hint 2</summary>

Loop `k` from 0 to `M-1`, then `for S in combinations(others, k)`: weight = `factorial(k) * factorial(M - k - 1) / factorial(M)`.

</details>

<details><summary>💡 Hint 3</summary>

Add `weight * (coalition_value(model, x, background, set(S) | {j}) - coalition_value(model, x, background, set(S)))`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [23]:
from math import factorial

def shapley_subsets(model, x, background, j):
    M = len(x)
    others = [i for i in range(M) if i != j]
    total = 0.0
    for k in range(M):
        w = factorial(k) * factorial(M - k - 1) / factorial(M)
        for S in combinations(others, k):
            total += w * (coalition_value(model, x, background, set(S) | {j}) - coalition_value(model, x, background, set(S)))
    return total

phi_dist = shapley_subsets(small, x, background, 1)

# ---- self-check ----
check("08.3", lambda: abs(phi_dist - phi_brute["distance_to_center_km"]) < 1e-6 and abs(shapley_subsets(small, x, background, 0) - phi_brute["size_sqm"]) < 1e-6,
      "Weights are |S|!(M-|S|-1)!/M!; the answer must equal the ordering-based value from the lesson.", needs=[phi_dist])

✅ 08.3: correct -- nice work!

**✅ Solution walk-through (Problem 08.3)**

Both formulations are the same number: grouping the 3! orderings by *which features come before j* yields exactly the subset weights |S|!(M−|S|−1)!/M!. The subset form is what TreeSHAP and KernelSHAP build on; the ordering form is easier to picture.

### 🧩 Problem 08.4 ⭐⭐: Hunt the leak

Use `leaky_model`, `Xl_test` and `y_test`:

* `leaky_feature`: the name of the feature with the **largest permutation importance** (use `perm_importance`),
* `leak_share`: that feature's importance divided by the **sum of all positive importances** (what fraction of the total does it hold?).

Then, in a markdown cell, write two sentences: *why is this feature a leak, and what would you ask the data owner?*

<details><summary>💡 Hint 1</summary>

`imp = perm_importance(leaky_model, Xl_test, y_test)` is already sorted descending; `imp.index[0]` is the top feature.

</details>

<details><summary>💡 Hint 2</summary>

`leak_share = imp.iloc[0] / imp[imp > 0].sum()`.

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [24]:
imp = perm_importance(leaky_model, Xl_test, y_test)
leaky_feature = imp.index[0]
leak_share = float(imp.iloc[0] / imp[imp > 0].sum())

# ---- self-check ----
check("08.4", lambda: leaky_feature == "agent_valuation" and leak_share > 0.5, "The leak should hold over half of the total importance.",
      needs=[leaky_feature, leak_share])

✅ 08.4: correct -- nice work!


**✅ Solution walk-through (Problem 08.4)**

`agent_valuation` holds ≈ 98% of all permutation importance (≈ 454 of ≈ 460 RMSE points). Question for the data owner: *'When is this valuation produced: before or after the sale price is known? Is it derived from the price?'* If it's only available after the outcome (or computed from it), it can't be used at prediction time and must be dropped; the honest model is the one without it (R² ≈ 0.94).

### 🧩 Problem 08.5 ⭐⭐: Top drivers of one prediction

Write `top_drivers(i, k)` that returns the `k` features with the **largest absolute SHAP contribution** for test house `i`, as a list of `(feature_name, shap_value)` pairs sorted by absolute size (largest first).

Then store `drivers_10 = top_drivers(10, 3)` and `biggest_up_10`: the name of the feature that pushes house #10's price **up** the most.

<details><summary>💡 Hint 1</summary>

`contrib = pd.Series(sv.values[i], index=feature_names)`.

</details>

<details><summary>💡 Hint 2</summary>

`contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(k)` sorts by magnitude while keeping the sign.

</details>

<details><summary>💡 Hint 3</summary>

`biggest_up_10 = contrib.idxmax()` (largest *positive* value).

</details>

<details><summary>🆘 Stuck for real? Read this</summary>

Walk away for five minutes, re-read the lesson section above, then compare with the solutions notebook of the same name (`../solutions/`). Peeking is allowed -- but *type* the solution yourself, don't paste it.

</details>


In [25]:
def top_drivers(i, k=3):
    contrib = pd.Series(sv.values[i], index=feature_names)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(k)
    return list(zip(top.index, top.values))

drivers_10 = top_drivers(10, 3)
biggest_up_10 = pd.Series(sv.values[10], index=feature_names).idxmax()

# ---- self-check ----
def _ref():
    c = pd.Series(sv.values[10], index=feature_names)
    return list(c.reindex(c.abs().sort_values(ascending=False).index).head(3).index), c.idxmax()
check("08.5", lambda: [d[0] for d in drivers_10] == _ref()[0] and biggest_up_10 == _ref()[1]
      and abs(drivers_10[0][1]) >= abs(drivers_10[1][1]) >= abs(drivers_10[2][1]) and len(top_drivers(3, 2)) == 2,
      "Sort by |SHAP| (keep the sign); biggest_up is the largest positive contribution.", needs=[drivers_10, biggest_up_10])

✅ 08.5: correct -- nice work!


**✅ Solution walk-through (Problem 08.5)**

Local explanations are just a row of the SHAP matrix, sorted. Turning `drivers_10` into a sentence with units and feature values (see `explain_in_words`) is the part that makes it useful to a human.

In [26]:
scoreboard()

Scoreboard: 5 solved, 0 to fix, 0 not attempted
   ✅ 08.1
   ✅ 08.2
   ✅ 08.3
   ✅ 08.4
   ✅ 08.5


## 🏁 Recap

* **Global** explanations describe the model overall; **local** ones explain a single prediction.
* **Built-in (MDI/gain) importance** is computed on training data and is biased toward continuous/high-cardinality features: we showed it giving noise columns real scores. Treat it as a sketch.
* **Permutation importance** (on held-out data) measures how much *error* grows when a feature is scrambled; beware **correlated features** (credit is masked): use grouped permutation.
* **PDP/ICE** show *how* predictions respond to a feature.
* **Shapley values** give each feature its average marginal contribution over all orderings; they **add up exactly** to prediction − base. **TreeSHAP** computes them exactly and fast for tree ensembles: we matched brute force to 4+ decimals.
* **SHAP plots:** bar (global size), beeswarm (size + direction), dependence (shape of effect, interactions), waterfall (one prediction). Our model **re-discovered the hidden recipe** (step, hinge, interaction, neighbourhood premiums).
* Interpretability is also a **debugging** tool (leaks!), but explanations describe the *model*, **not causation**.

### ➡️ Next: **Lesson 09: The Practical Assignment**
Everything converges: five regression models on one dataset, honest evaluation, and an evidence-based explanation of *why* one wins. Then the **Capstone Project**.